# Grok-rl-09-imitation-offline

**Stage 09 — Imitation Learning & Offline RL**

## 概念
很多机器人场景 **不能在线乱探索**（昂贵/危险）。

1. **Behavior Cloning (BC)** — 监督学习模仿专家动作
2. **Offline RL** — 仅用固定数据集优化策略；朴素 off-policy 会 **OOD Q 高估**
3. **CQL-lite** — 对数据集外动作的 Q 加保守惩罚

## 环境
Cliff grid（离散）+ 专家 = 最优短路径策略；数据集含专家 + 噪声轨迹。


In [ ]:

import json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
SEED=0; np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpu={"cuda":torch.cuda.is_available(),"device_count":torch.cuda.device_count(),"names":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print(gpu)

ACTS={0:(-1,0),1:(0,1),2:(1,0),3:(0,-1)}
class Cliff:
    def __init__(self):
        self.H,self.W=4,12; self.start=(3,0); self.goal=(3,11)
        self.cliff={(3,c) for c in range(1,11)}; self.nS=48; self.nA=4
    def sid(self,r,c): return r*self.W+c
    def reset(self):
        self.s=self.start; return self.sid(*self.s)
    def step(self,a):
        r,c=self.s; dr,dc=ACTS[a]; nr,nc=r+dr,c+dc
        if not(0<=nr<self.H and 0<=nc<self.W): nr,nc=r,c
        if (nr,nc) in self.cliff:
            self.s=self.start; return self.sid(*self.s), -100., True
        if (nr,nc)==self.goal:
            self.s=(nr,nc); return self.sid(*self.s), 10., True
        self.s=(nr,nc); return self.sid(*self.s), -1., False

# Expert: go up, right along top, down near goal
def expert_action(s, env):
    r,c=divmod(s, env.W)
    if r>0 and c<11:  # go up first if not on top row carefully
        if r==3 and c==0: return 0  # up
    if r>0 and c==0: return 0
    if r==0 and c<11: return 1  # right on top
    if c==11 and r<3: return 2  # down
    return 1

def gen_dataset(env, n_traj=200, noise=0.2):
    S,A,R,NS,D=[],[],[],[],[]
    for _ in range(n_traj):
        s=env.reset(); done=False; steps=0
        while not done and steps<100:
            if np.random.rand()<noise:
                a=np.random.randint(0,4)
            else:
                a=expert_action(s, env)
            ns,r,done=env.step(a)
            S.append(s); A.append(a); R.append(r); NS.append(ns); D.append(float(done))
            s=ns; steps+=1
    return map(np.array, (S,A,R,NS,D))


In [ ]:

env=Cliff()
S,A,R,NS,D=gen_dataset(env, n_traj=300, noise=0.25)
print("transitions", len(S), "mean r", R.mean())

# one-hot state embedding
def oh(s):
    x=np.zeros((len(s), env.nS), np.float32); x[np.arange(len(s)), s]=1; return x

class PiNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(env.nS,64),nn.ReLU(),nn.Linear(64,4))
    def forward(self,x): return self.net(x)

class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(env.nS,64),nn.ReLU(),nn.Linear(64,4))
    def forward(self,x): return self.net(x)

def train_bc(steps=2000):
    pi=PiNet().to(device)
    opt=torch.optim.Adam(pi.parameters(), lr=1e-3)
    X=torch.tensor(oh(S),device=device); y=torch.tensor(A,device=device)
    for _ in range(steps):
        idx=torch.randint(0,len(S),(128,),device=device)
        loss=F.cross_entropy(pi(X[idx]), y[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    return pi

def train_online_q_naive_offline(steps=3000, cql_alpha=0.0):
    # offline DQN +/- CQL regularizer
    q=QNet().to(device); qt=QNet().to(device); qt.load_state_dict(q.state_dict())
    opt=torch.optim.Adam(q.parameters(), lr=1e-3)
    X=torch.tensor(oh(S),device=device); XA=torch.tensor(A,device=device)
    XR=torch.tensor(R,device=device,dtype=torch.float32); XN=torch.tensor(oh(NS),device=device); XD=torch.tensor(D,device=device,dtype=torch.float32)
    for t in range(steps):
        idx=torch.randint(0,len(S),(128,),device=device)
        qs=q(X[idx]).gather(1, XA[idx].unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            y=XR[idx] + (1-XD[idx])*0.99*qt(XN[idx]).max(1)[0]
        td=F.mse_loss(qs,y)
        # CQL-lite: logsumexp all actions - data actions
        allq=q(X[idx])
        cql=torch.logsumexp(allq, dim=1).mean() - qs.mean()
        loss=td + cql_alpha*cql
        opt.zero_grad(); loss.backward(); opt.step()
        if t%50==0: qt.load_state_dict(q.state_dict())
    return q

def eval_pi_from_logits(net, n=30):
    scores=[]
    for _ in range(n):
        s=env.reset(); R=0; done=False; steps=0
        while not done and steps<100:
            with torch.no_grad():
                x=torch.tensor(oh([s]),device=device)
                a=int(net(x).argmax(1).item())
            s,r,done=env.step(a); R+=r; steps+=1
        scores.append(R)
    return float(np.mean(scores))

def eval_q_greedy(q, n=30):
    scores=[]
    for _ in range(n):
        s=env.reset(); R=0; done=False; steps=0
        while not done and steps<100:
            with torch.no_grad():
                x=torch.tensor(oh([s]),device=device)
                a=int(q(x).argmax(1).item())
            s,r,done=env.step(a); R+=r; steps+=1
        scores.append(R)
    return float(np.mean(scores))

t0=time.time()
pi=train_bc()
q_naive=train_online_q_naive_offline(cql_alpha=0.0)
q_cql=train_online_q_naive_offline(cql_alpha=1.0)
bc_sc=eval_pi_from_logits(pi)
naive_sc=eval_q_greedy(q_naive)
cql_sc=eval_q_greedy(q_cql)
# expert score
exp_scores=[]
for _ in range(30):
    s=env.reset(); R=0; done=False; steps=0
    while not done and steps<100:
        a=expert_action(s,env); s,r,done=env.step(a); R+=r; steps+=1
    exp_scores.append(R)
exp_sc=float(np.mean(exp_scores))
elapsed=time.time()-t0
print("expert", exp_sc, "BC", bc_sc, "offline-DQN", naive_sc, "CQL", cql_sc)


In [ ]:

fig,ax=plt.subplots(figsize=(7,4))
names=["Expert","BC","Offline DQN","CQL-lite"]
vals=[exp_sc, bc_sc, naive_sc, cql_sc]
ax.bar(names, vals, color=["#333","#2a9d8f","#e76f51","#264653"])
ax.set_ylabel("mean return"); ax.set_title("Imitation vs Offline RL on Cliff dataset")
fig.tight_layout(); fig.savefig(OUT/"stage09_imitation_offline.png", dpi=120); plt.close(fig)

payload={
  "ok": True,
  "stage":"09-imitation-offline",
  "title":"Grok-rl-09-imitation-offline",
  "metrics":{"expert":exp_sc,"bc":bc_sc,"offline_dqn":naive_sc,"cql":cql_sc},
  "gpu":gpu,"elapsed_sec":elapsed,
  "concept": "learn from fixed demos without online exploration; CQL conservatively regularizes Q",
  "new_capability": "deployable policies from logged robot data",
  "compare_to_previous": "Stages07-08 needed online interaction; Stage09 only uses offline dataset",
}
assert payload["metrics"]["bc"] > -50  # BC should mostly work
(OUT/"results_stage09.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2))
print("STAGE09_OK")
